In [ ]:
import torch
import pandas as pd

from dlem.dlem_genome import dlem_notebook

cooler_file = '/Users/tina/LoopExtrusion/data/H1.mcool'
output_path = f'/Users/tina/LoopExtrusion/results/chr10_test.tsv'
processed_output_path = f'/Users/tina/LoopExtrusion/results/chr10_test_processed.tsv'
resolution = 10_000
window_size = 200
stride = 50
dev_name ='cpu'

In [ ]:

def process_dataframe(df_result):
    """
    Process the dataframe to create wide format with weighted means.
    
    Parameters:
    df_result: The input dataframe with columns chrom, start, end, patch_i, max_corr, 
               perc_nan, importance_weights, left, right
    
    Returns:
    DataFrame: Processed dataframe with weighted averages
    """
    # Create a location identifier based on unique start/end combinations
    df_result['location_id'] = df_result.groupby(['start', 'end']).ngroup()
    
    # Create instance_id for each row within a location group
    df_result['instance_id'] = df_result.groupby('location_id').cumcount() + 1
    
    # Define columns that should have multiple instances
    varying_columns = ['patch_i', 'max_corr', 'perc_nan', 'importance_weights', 'left', 'right']
    
    # Pivot to create wide format for varying columns
    wide_df = pd.pivot(
        index='location_id',
        columns='instance_id',
        values=varying_columns,
        data=df_result
    )
    
    # Flatten the column names
    wide_df.columns = [f'{col[0]}_{col[1]}' for col in wide_df.columns]
    
    # Add the constant columns (start, end, chrom)
    constant_df = df_result.drop_duplicates('location_id')[['location_id', 'start', 'end', 'chrom']]
    
    # Join the constant columns with the wide format
    final_df = pd.merge(constant_df, wide_df, on='location_id')
    
    # Identify importance_weights columns
    importance_weight_cols = [col for col in final_df.columns if 'importance_weights_' in col]
    
    # Replace NaN values with 0 in all importance_weights columns
    final_df[importance_weight_cols] = final_df[importance_weight_cols].fillna(0)
    
    # Calculate sum of importance weights
    final_df['sum_importance_weights'] = final_df[importance_weight_cols].sum(axis=1)
    
    # Create weighted columns for left
    weighted_left_cols = []
    for i in range(1, 5):
        left_col = f'left_{i}'
        weight_col = f'importance_weights_{i}'
        weighted_col = f'weighted_left_{i}'
        weighted_left_cols.append(weighted_col)
        
        # If left is NaN, set weighted value to 0
        final_df[weighted_col] = np.where(
            final_df[left_col].isna(),
            0,
            final_df[left_col] * final_df[weight_col]
        )
    
    # Create weighted columns for right
    weighted_right_cols = []
    for i in range(1, 5):
        right_col = f'right_{i}'
        weight_col = f'importance_weights_{i}'
        weighted_col = f'weighted_right_{i}'
        weighted_right_cols.append(weighted_col)
        
        # If right is NaN, set weighted value to 0
        final_df[weighted_col] = np.where(
            final_df[right_col].isna(),
            0,
            final_df[right_col] * final_df[weight_col]
        )
    
    # Calculate sum of weighted values
    final_df['weighted_left_sum'] = final_df[weighted_left_cols].sum(axis=1)
    final_df['weighted_right_sum'] = final_df[weighted_right_cols].sum(axis=1)
    
    # Calculate weighted averages
    final_df['weighted_avg_left'] = final_df.apply(
        lambda row: row['weighted_left_sum'] / row['sum_importance_weights'] 
                   if row['sum_importance_weights'] > 0 else 0, 
        axis=1
    )
    
    final_df['weighted_avg_right'] = final_df.apply(
        lambda row: row['weighted_right_sum'] / row['sum_importance_weights'] 
                   if row['sum_importance_weights'] > 0 else 0, 
        axis=1
    )
    
    # Drop location_id if not needed
    final_df = final_df.drop(columns=['location_id'])

    final_df = final_df.set_index('start')
    

    final_df = final_df[['end', 'chrom', 'weighted_avg_left', 'weighted_avg_right']]

    final_df.rename(columns={'weighted_avg_left': 'left', 'weighted_avg_right': 'right'}, inplace=True)

    return final_df

# Example usage:
# final_processed_df = process_dataframe(df_result)
# print(final_processed_df.head())

In [ ]:
df_result = dlem_notebook(cooler_file, output_path, resolution, 
                          model_name="minimal_dlem", window_size=window_size, stride=stride,
                          chrom_subset=['chr10'], perc_nan_threshold=0.3, lr=0.5, 
                          reader_name='datareader_cooler', dev_name=dev_name, do_return_result=True)

final_df = process_dataframe(df_result)
final_df_reset_index = final_df.reset_index()

final_df_reset_index.to_csv(processed_output_path , sep='\t')